# Privacy-Preserving Data with Differential Privacy: DP-SGD on UCI Adult Dataset

Notebook này tái hiện ý tưởng chính của bài báo Deep Learning with Differential Privacy theo hướng dữ liệu bảng.

Nội dung chính:

1. Tải UCI Adult Dataset
2. Tiền xử lý dữ liệu dạng bảng
3. Train mô hình baseline không áp dụng Differential Privacy
4. Train mô hình MLP với DP-SGD bằng Opacus
5. Chạy nhiều mức noise multiplier
6. Theo dõi epsilon theo epoch
7. So sánh accuracy, precision, recall, F1-score, training time
8. Vẽ privacy-utility tradeoff giữa epsilon và accuracy

Dataset: UCI Adult Dataset  
Task: Dự đoán thu nhập >50K hoặc <=50K  
Model: MLP  
DP library: Opacus

In [ ]:
# ============================================================
# Cell 1. Install required libraries
# ============================================================

!pip install -q opacus ucimlrepo

In [ ]:
# ============================================================
# Cell 2. Import libraries
# ============================================================

import os
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from ucimlrepo import fetch_ucirepo
from opacus import PrivacyEngine

warnings.filterwarnings("ignore")

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# ============================================================
# Cell 3. Experimental configuration
# Inspired by Abadi et al. Deep Learning with Differential Privacy
# Goal: compare non-private baseline with DP-SGD under multiple noise levels
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Training settings
BATCH_SIZE = 256
EPOCHS = 20
LR_BASELINE = 0.001
LR_DP = 0.05

# Differential Privacy settings
DELTA = 1e-5
MAX_GRAD_NORM = 1.0

# Multiple noise levels for privacy-utility tradeoff
NOISE_LIST = [0.5, 0.8, 1.0, 1.5, 2.0, 3.0]

# Reproducibility
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if DEVICE == "cuda":
    torch.cuda.manual_seed_all(RANDOM_STATE)

print("Device:", DEVICE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Delta:", DELTA)
print("Max grad norm:", MAX_GRAD_NORM)
print("Noise list:", NOISE_LIST)

## 1. Tải và kiểm tra UCI Adult Dataset

UCI Adult Dataset là bài toán phân loại nhị phân. Mục tiêu là dự đoán một người có thu nhập >50K hay <=50K dựa trên các thuộc tính cá nhân như tuổi, học vấn, nghề nghiệp, tình trạng hôn nhân, giới tính và số giờ làm việc mỗi tuần.

In [ ]:
# ============================================================
# Cell 4. Load UCI Adult Dataset
# ============================================================

adult = fetch_ucirepo(id=2)

X = adult.data.features.copy()
y = adult.data.targets.copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())
display(y.head())

In [ ]:
# ============================================================
# Cell 5. Inspect raw dataset
# ============================================================

print("Feature columns:")
print(X.columns.tolist())

print("\nTarget columns:")
print(y.columns.tolist())

print("\nMissing values in X:")
print(X.isna().sum())

print("\nTarget value counts:")
print(y.iloc[:, 0].value_counts(dropna=False))

In [ ]:
# ============================================================
# Cell 6. Clean data and normalize labels
# ============================================================

df = X.copy()
df["income"] = y.iloc[:, 0]

# Normalize target labels
df["income"] = df["income"].astype(str).str.strip()
df["income"] = df["income"].str.replace(".", "", regex=False)

# Replace question mark values with NaN, then drop missing rows
df = df.replace("?", np.nan)
df = df.dropna().reset_index(drop=True)

# Encode target: >50K = 1, <=50K = 0
df["target"] = df["income"].apply(lambda v: 1 if v == ">50K" else 0)
df = df.drop(columns=["income"])

print("Cleaned shape:", df.shape)
print("\nTarget distribution:")
print(df["target"].value_counts())
print("\nTarget ratio:")
print(df["target"].value_counts(normalize=True))

display(df.head())

## 2. Tiền xử lý dữ liệu bảng

Các biến số được chuẩn hóa bằng StandardScaler. Các biến categorical được biến đổi bằng one-hot encoding.

In [ ]:
# ============================================================
# Cell 7. Split features and target
# ============================================================

X_df = df.drop(columns=["target"])
y_array = df["target"].values.astype(np.int64)

categorical_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X_df.select_dtypes(exclude=["object", "category"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

In [ ]:
# ============================================================
# Cell 8. Train/test split
# ============================================================

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_df,
    y_array,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_array
)

print("Train shape:", X_train_df.shape)
print("Test shape:", X_test_df.shape)

print("\nTrain target distribution:")
print(pd.Series(y_train).value_counts(normalize=True))

print("\nTest target distribution:")
print(pd.Series(y_test).value_counts(normalize=True))

In [ ]:
# ============================================================
# Cell 9. One-hot encoding and standardization
# ============================================================

# Compatible with both old and new scikit-learn versions
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", encoder, categorical_cols)
    ]
)

X_train = preprocessor.fit_transform(X_train_df)
X_test = preprocessor.transform(X_test_df)

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

INPUT_DIM = X_train.shape[1]

print("Processed train shape:", X_train.shape)
print("Processed test shape:", X_test.shape)
print("Input dim:", INPUT_DIM)

In [ ]:
# ============================================================
# Cell 10. Create TensorDataset and DataLoader
# ============================================================

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))

## 3. Xây dựng mô hình MLP

Vì UCI Adult là dữ liệu dạng bảng, notebook dùng MLP thay vì CNN. Đây là điểm khác với bài báo gốc, nhưng lõi DP-SGD vẫn giống: clipping gradient, thêm Gaussian noise và theo dõi privacy budget.

In [ ]:
# ============================================================
# Cell 11. Define MLP model
# ============================================================

class AdultMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)


model_check = AdultMLP(INPUT_DIM)
print(model_check)
print("Number of parameters:", sum(p.numel() for p in model_check.parameters()))

In [ ]:
# ============================================================
# Cell 12. Training and evaluation functions
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_batches = 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

    return total_loss / total_batches


def evaluate_model(model, loader, criterion=None):
    model.eval()

    all_preds = []
    all_labels = []
    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(DEVICE)
            y_batch_device = y_batch.to(DEVICE)

            logits = model(x_batch)

            if criterion is not None:
                loss = criterion(logits, y_batch_device)
                total_loss += loss.item()
                total_batches += 1

            preds = torch.argmax(logits, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y_batch.numpy())

    avg_loss = total_loss / total_batches if total_batches > 0 else None

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "y_true": np.array(all_labels),
        "y_pred": np.array(all_preds)
    }

## 4. Train baseline không áp dụng Differential Privacy

Baseline dùng Adam optimizer và không có gradient clipping/noise theo cơ chế DP.

In [ ]:
# ============================================================
# Cell 13. Train non-private baseline
# ============================================================

def run_baseline_experiment():
    model = AdultMLP(INPUT_DIM).to(DEVICE)

    optimizer = optim.Adam(model.parameters(), lr=LR_BASELINE)
    criterion = nn.CrossEntropyLoss()

    history = []
    start_time = time.time()

    print("=" * 80)
    print("Training non-private baseline model")
    print("=" * 80)

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion
        )

        eval_result = evaluate_model(
            model=model,
            loader=test_loader,
            criterion=criterion
        )

        epoch_result = {
            "method": "Baseline",
            "epoch": epoch,
            "noise_multiplier": 0.0,
            "epsilon": np.nan,
            "delta": np.nan,
            "train_loss": train_loss,
            "test_loss": eval_result["loss"],
            "accuracy": eval_result["accuracy"],
            "precision": eval_result["precision"],
            "recall": eval_result["recall"],
            "f1": eval_result["f1"]
        }

        history.append(epoch_result)

        print(
            f"Baseline Epoch {epoch:02d}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Test Loss: {eval_result['loss']:.4f} | "
            f"Acc: {eval_result['accuracy']:.4f} | "
            f"F1: {eval_result['f1']:.4f}"
        )

    training_time = time.time() - start_time
    final_eval = evaluate_model(model, test_loader, criterion)

    print("\nClassification Report:")
    print(classification_report(final_eval["y_true"], final_eval["y_pred"], zero_division=0))

    print("\nConfusion Matrix:")
    print(confusion_matrix(final_eval["y_true"], final_eval["y_pred"]))

    final_result = {
        "method": "Baseline",
        "noise_multiplier": 0.0,
        "epsilon": np.nan,
        "delta": np.nan,
        "accuracy": final_eval["accuracy"],
        "precision": final_eval["precision"],
        "recall": final_eval["recall"],
        "f1": final_eval["f1"],
        "training_time": training_time
    }

    print("\nFinal Baseline Result")
    print(final_result)

    return model, final_result, pd.DataFrame(history)


baseline_model, baseline_result, baseline_history = run_baseline_experiment()

## 5. Train DP-SGD với nhiều mức noise multiplier

Mỗi lần train DP-SGD sẽ:

1. Tính per-example gradient bằng Opacus
2. Clip gradient theo max_grad_norm
3. Thêm Gaussian noise theo noise_multiplier
4. Cập nhật trọng số
5. Tính epsilon sau từng epoch

In [ ]:
# ============================================================
# Cell 14. Define DP-SGD experiment function
# ============================================================

def run_dp_sgd_experiment(noise_multiplier):
    model = AdultMLP(INPUT_DIM).to(DEVICE)

    optimizer = optim.SGD(model.parameters(), lr=LR_DP, momentum=0.9)
    criterion = nn.CrossEntropyLoss()

    privacy_engine = PrivacyEngine(accountant="prv")

    model, optimizer, private_train_loader = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=train_loader,
        noise_multiplier=noise_multiplier,
        max_grad_norm=MAX_GRAD_NORM
    )

    history = []
    start_time = time.time()

    print("=" * 80)
    print(f"Training DP-SGD model | noise_multiplier = {noise_multiplier}")
    print("=" * 80)

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=private_train_loader,
            optimizer=optimizer,
            criterion=criterion
        )

        eval_result = evaluate_model(
            model=model,
            loader=test_loader,
            criterion=criterion
        )

        epsilon = privacy_engine.get_epsilon(DELTA)

        epoch_result = {
            "method": "DP-SGD",
            "epoch": epoch,
            "noise_multiplier": noise_multiplier,
            "epsilon": epsilon,
            "delta": DELTA,
            "train_loss": train_loss,
            "test_loss": eval_result["loss"],
            "accuracy": eval_result["accuracy"],
            "precision": eval_result["precision"],
            "recall": eval_result["recall"],
            "f1": eval_result["f1"]
        }

        history.append(epoch_result)

        print(
            f"DP Epoch {epoch:02d}/{EPOCHS} | "
            f"Noise: {noise_multiplier:.2f} | "
            f"Epsilon: {epsilon:.4f} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Test Loss: {eval_result['loss']:.4f} | "
            f"Acc: {eval_result['accuracy']:.4f} | "
            f"F1: {eval_result['f1']:.4f}"
        )

    training_time = time.time() - start_time
    final_eval = evaluate_model(model, test_loader, criterion)
    final_epsilon = privacy_engine.get_epsilon(DELTA)

    print("\nClassification Report:")
    print(classification_report(final_eval["y_true"], final_eval["y_pred"], zero_division=0))

    print("\nConfusion Matrix:")
    print(confusion_matrix(final_eval["y_true"], final_eval["y_pred"]))

    final_result = {
        "method": "DP-SGD",
        "noise_multiplier": noise_multiplier,
        "epsilon": final_epsilon,
        "delta": DELTA,
        "accuracy": final_eval["accuracy"],
        "precision": final_eval["precision"],
        "recall": final_eval["recall"],
        "f1": final_eval["f1"],
        "training_time": training_time
    }

    print("\nFinal DP-SGD Result")
    print(final_result)

    return model, final_result, pd.DataFrame(history)

In [ ]:
# ============================================================
# Cell 15. Run all DP-SGD experiments with multiple noise levels
# ============================================================

all_final_results = []
all_epoch_histories = []

# Add baseline result
all_final_results.append(baseline_result)
all_epoch_histories.append(baseline_history)

dp_models = {}

for noise in NOISE_LIST:
    dp_model, dp_result, dp_history = run_dp_sgd_experiment(noise_multiplier=noise)

    dp_models[noise] = dp_model
    all_final_results.append(dp_result)
    all_epoch_histories.append(dp_history)

df_results = pd.DataFrame(all_final_results)
df_history = pd.concat(all_epoch_histories, ignore_index=True)

display(df_results)
display(df_history.head())

## 6. So sánh kết quả cuối cùng

In [ ]:
# ============================================================
# Cell 16. Final comparison table
# ============================================================

df_compare = df_results.copy()

df_compare["accuracy_drop_vs_baseline"] = np.nan
df_compare["f1_drop_vs_baseline"] = np.nan
df_compare["time_ratio_vs_baseline"] = np.nan

baseline_acc = df_compare.loc[df_compare["method"] == "Baseline", "accuracy"].iloc[0]
baseline_f1 = df_compare.loc[df_compare["method"] == "Baseline", "f1"].iloc[0]
baseline_time = df_compare.loc[df_compare["method"] == "Baseline", "training_time"].iloc[0]

for idx in df_compare.index:
    df_compare.loc[idx, "accuracy_drop_vs_baseline"] = baseline_acc - df_compare.loc[idx, "accuracy"]
    df_compare.loc[idx, "f1_drop_vs_baseline"] = baseline_f1 - df_compare.loc[idx, "f1"]
    df_compare.loc[idx, "time_ratio_vs_baseline"] = df_compare.loc[idx, "training_time"] / baseline_time

df_compare = df_compare[
    [
        "method",
        "noise_multiplier",
        "epsilon",
        "delta",
        "accuracy",
        "accuracy_drop_vs_baseline",
        "precision",
        "recall",
        "f1",
        "f1_drop_vs_baseline",
        "training_time",
        "time_ratio_vs_baseline"
    ]
]

display(df_compare)

df_compare.to_csv("uci_adult_dp_sgd_final_comparison.csv", index=False)
df_history.to_csv("uci_adult_dp_sgd_epoch_history.csv", index=False)

print("Saved:")
print("uci_adult_dp_sgd_final_comparison.csv")
print("uci_adult_dp_sgd_epoch_history.csv")

## 7. Biểu đồ privacy-utility tradeoff

Biểu đồ quan trọng nhất là epsilon - accuracy. Đây là phần giống tinh thần bài báo gốc nhất.

In [ ]:
# ============================================================
# Cell 17. Privacy-Utility Tradeoff: epsilon vs accuracy
# ============================================================

dp_df = df_compare[df_compare["method"] == "DP-SGD"].copy()
dp_df = dp_df.sort_values("epsilon")

plt.figure(figsize=(8, 5))
plt.plot(dp_df["epsilon"], dp_df["accuracy"], marker="o")
plt.xlabel("Privacy Budget epsilon")
plt.ylabel("Test Accuracy")
plt.title("Privacy-Utility Tradeoff on UCI Adult Dataset")
plt.grid(True)
plt.tight_layout()
plt.savefig("privacy_utility_epsilon_accuracy.png", dpi=200)
plt.show()

print("Saved: privacy_utility_epsilon_accuracy.png")

In [ ]:
# ============================================================
# Cell 18. Noise multiplier vs epsilon
# ============================================================

dp_df = df_compare[df_compare["method"] == "DP-SGD"].copy()
dp_df = dp_df.sort_values("noise_multiplier")

plt.figure(figsize=(8, 5))
plt.plot(dp_df["noise_multiplier"], dp_df["epsilon"], marker="o")
plt.xlabel("Noise Multiplier")
plt.ylabel("Privacy Budget epsilon")
plt.title("Effect of Noise on Privacy Budget")
plt.grid(True)
plt.tight_layout()
plt.savefig("noise_vs_epsilon.png", dpi=200)
plt.show()

print("Saved: noise_vs_epsilon.png")

In [ ]:
# ============================================================
# Cell 19. Noise multiplier vs accuracy
# ============================================================

dp_df = df_compare[df_compare["method"] == "DP-SGD"].copy()
dp_df = dp_df.sort_values("noise_multiplier")

plt.figure(figsize=(8, 5))
plt.plot(dp_df["noise_multiplier"], dp_df["accuracy"], marker="o")
plt.xlabel("Noise Multiplier")
plt.ylabel("Test Accuracy")
plt.title("Effect of DP Noise on Accuracy")
plt.grid(True)
plt.tight_layout()
plt.savefig("noise_vs_accuracy.png", dpi=200)
plt.show()

print("Saved: noise_vs_accuracy.png")

In [ ]:
# ============================================================
# Cell 20. Baseline vs DP-SGD accuracy comparison
# ============================================================

plot_df = df_compare.copy()

plot_df["label"] = plot_df.apply(
    lambda row: "Baseline" if row["method"] == "Baseline"
    else f"DP noise={row['noise_multiplier']}",
    axis=1
)

plt.figure(figsize=(10, 5))
plt.bar(plot_df["label"], plot_df["accuracy"])
plt.xlabel("Experiment")
plt.ylabel("Test Accuracy")
plt.title("Baseline vs DP-SGD Accuracy")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.tight_layout()
plt.savefig("baseline_vs_dp_accuracy.png", dpi=200)
plt.show()

print("Saved: baseline_vs_dp_accuracy.png")

In [ ]:
# ============================================================
# Cell 21. Epsilon over training epochs
# ============================================================

dp_history = df_history[df_history["method"] == "DP-SGD"].copy()

plt.figure(figsize=(9, 5))

for noise in sorted(dp_history["noise_multiplier"].unique()):
    temp = dp_history[dp_history["noise_multiplier"] == noise]
    plt.plot(temp["epoch"], temp["epsilon"], marker="o", label=f"noise={noise}")

plt.xlabel("Epoch")
plt.ylabel("Privacy Budget epsilon")
plt.title("Privacy Budget Accumulation During DP-SGD Training")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("epsilon_over_epochs.png", dpi=200)
plt.show()

print("Saved: epsilon_over_epochs.png")

In [ ]:
# ============================================================
# Cell 22. Accuracy over training epochs
# ============================================================

plt.figure(figsize=(9, 5))

baseline_hist = df_history[df_history["method"] == "Baseline"]
plt.plot(
    baseline_hist["epoch"],
    baseline_hist["accuracy"],
    marker="o",
    label="Baseline"
)

dp_history = df_history[df_history["method"] == "DP-SGD"]

for noise in sorted(dp_history["noise_multiplier"].unique()):
    temp = dp_history[dp_history["noise_multiplier"] == noise]
    plt.plot(
        temp["epoch"],
        temp["accuracy"],
        marker="o",
        label=f"DP noise={noise}"
    )

plt.xlabel("Epoch")
plt.ylabel("Test Accuracy")
plt.title("Accuracy During Training")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("accuracy_over_epochs.png", dpi=200)
plt.show()

print("Saved: accuracy_over_epochs.png")

## 8. Tự sinh nhận xét kết quả

Cell cuối cùng tự tạo phần nhận xét để đưa vào báo cáo.

In [ ]:
# ============================================================
# Cell 23. Auto-generate experiment summary
# ============================================================

dp_df = df_compare[df_compare["method"] == "DP-SGD"].copy()

best_dp_by_accuracy = dp_df.sort_values("accuracy", ascending=False).iloc[0]
best_dp_by_privacy = dp_df.sort_values("epsilon", ascending=True).iloc[0]
baseline_row = df_compare[df_compare["method"] == "Baseline"].iloc[0]

summary_text = f'''
Nhận xét kết quả thí nghiệm:

Mô hình baseline không áp dụng Differential Privacy đạt accuracy = {baseline_row['accuracy']:.4f}, F1-score = {baseline_row['f1']:.4f}, thời gian train = {baseline_row['training_time']:.2f} giây.

Khi áp dụng DP-SGD với nhiều mức noise multiplier khác nhau, privacy budget epsilon và accuracy thay đổi theo từng cấu hình. Cấu hình DP-SGD có accuracy cao nhất là noise_multiplier = {best_dp_by_accuracy['noise_multiplier']}, đạt accuracy = {best_dp_by_accuracy['accuracy']:.4f}, F1-score = {best_dp_by_accuracy['f1']:.4f}, epsilon = {best_dp_by_accuracy['epsilon']:.4f}. So với baseline, accuracy giảm {best_dp_by_accuracy['accuracy_drop_vs_baseline']:.4f}.

Cấu hình có mức bảo vệ riêng tư mạnh nhất là noise_multiplier = {best_dp_by_privacy['noise_multiplier']}, với epsilon = {best_dp_by_privacy['epsilon']:.4f}, accuracy = {best_dp_by_privacy['accuracy']:.4f}. Điều này cho thấy khi noise multiplier tăng, epsilon thường giảm, tức là mức bảo vệ quyền riêng tư mạnh hơn. Tuy nhiên, việc thêm nhiều nhiễu hơn có thể làm giảm hiệu năng mô hình.

Nhìn chung, kết quả thể hiện đúng privacy-utility tradeoff: DP-SGD giúp giới hạn ảnh hưởng của từng cá nhân trong tập huấn luyện thông qua gradient clipping và Gaussian noise, nhưng có thể làm accuracy giảm so với mô hình baseline. Với UCI Adult Dataset, mức giảm accuracy trong thí nghiệm này có thể được đánh giá là chi phí hiệu năng để đổi lấy bảo vệ quyền riêng tư.
'''

print(summary_text)

with open("uci_adult_dp_sgd_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

print("Saved: uci_adult_dp_sgd_summary.txt")

## 9. Gợi ý viết phần so sánh pipeline với bài báo gốc

Bài báo gốc Deep Learning with Differential Privacy của Abadi và cộng sự đề xuất pipeline huấn luyện DP-SGD cho neural network. Pipeline gồm các bước chính: xây dựng mô hình baseline, tính gradient riêng cho từng mẫu dữ liệu, clipping gradient để giới hạn ảnh hưởng của từng cá nhân, thêm nhiễu Gaussian vào gradient trung bình, cập nhật mô hình và sử dụng privacy accountant để theo dõi privacy budget epsilon.

Trong bài báo gốc, tác giả thực nghiệm trên MNIST và CIFAR-10, đồng thời thay đổi các mức noise để phân tích sự đánh đổi giữa privacy và accuracy. Các tham số như noise level, clipping norm, lot size và learning rate được xem là những yếu tố quan trọng ảnh hưởng đến kết quả.

Trong thí nghiệm này, pipeline được điều chỉnh sang UCI Adult Dataset. Thay vì xử lý ảnh, dữ liệu dạng bảng được tiền xử lý bằng chuẩn hóa biến số và one-hot encoding biến phân loại. Mô hình sử dụng là MLP để dự đoán nhóm thu nhập. Tương tự bài báo gốc, thí nghiệm gồm hai phần: train baseline không áp dụng Differential Privacy và train mô hình DP-SGD bằng Opacus. Kết quả được so sánh theo accuracy, epsilon, delta và thời gian train. Để bám sát bài báo gốc hơn, thí nghiệm bổ sung nhiều mức noise multiplier nhằm vẽ đường cong privacy-utility giữa epsilon và accuracy.